# Complete Guide to Linear Regression: The Taxi Fare Case Study

This notebook is a comprehensive guide to **Linear Regression**, following the official [Google Machine Learning Crash Course](https://developers.google.com/machine-learning/crash-course/linear-regression).

We will use the real-world problem of **predicting taxi fares** to explain every core concept of Machine Learning, from simple lines to gradient descent optimization.

---

## 🚕 1. The Core Concepts: What is Linear Regression?

**Linear Regression** is a method to find the best relationship between a **Label** (what we want to predict) and one or more **Features** (what we know).

### The Taxi Example:
- **Label (y)**: The `FARE` ($) of the trip.
- **Feature (x)**: The `TRIP_MILES` (distance traveled).

### The Mathematical Model
We represent this relationship as a straight line equation:
**$$y = wx + b$$**

1. **$y'$ (y-prime)**: The predicted value (the Fare we guess).
2. **$w$ (Weight)**: The slope. In our case, this is the **Price per Mile**.
3. **$x$**: The input. This is the **Distance (Miles)**.
4. **$b$ (Bias)**: The y-intercept. This is the **Base Fee** (the cost just for getting in the taxi).

**Training** a model simply means finding the "perfect" values for $w$ and $b$ that best fit our historical taxi data.

## 📉 2. Measuring Success: The Loss Function

In Machine Learning, we need a mathematical way to tell the model, "You are wrong by this much." We call this **Loss**.

### L2 Loss (Squared Error)
The standard loss for regression is **Squared Loss** (also called **L2 Loss**). For a single trip:
$$Loss = (y_{actual} - y_{predicted})^2$$

### Mean Squared Error (MSE)
To measure how the model is doing on *all* trips, we take the average of all individual losses. This is **Mean Squared Error (MSE)**.

**Original Formula:**
$$MSE = \frac{1}{N} \sum_{(x,y) \in D} (y - (wx + b))^2$$

- **Why square it?** Squaring removes negative signs (so an over-prediction doesn't cancel out an under-prediction) and it punishes large errors much more than small ones. If you are off by $2, the loss is 4. If you are off by $10, the loss is 100!

## 🔄 3. Optimization: Gradient Descent

How does the model actually "learn"? It uses an iterative approach called **Gradient Descent** to reduce the loss.

### The "Foggy Mountain" Analogy
Imagine you are at the top of a mountain (High Loss) in thick fog. You want to reach the bottom (Low Loss).
1. **Calculate Gradient**: You feel the slope with your feet to see which way is "down."
2. **Step Down**: You take a small step in that direction.
3. **Repeat**: You keep doing this until the ground is flat (Convergence).

### Deriving the Gradient (The "Fix")
To know the slope, we take the derivative of the MSE formula with respect to the weight ($w$):
1. Start with $Loss = (y - (wx + b))^2$
2. Use the **Chain Rule**: The derivative of $(\text{something})^2$ is $2 \times (\text{something}) \times (\text{derivative of inner part})$.
3. The derivative of $(y - wx - b)$ with respect to $w$ is $-x$.
4. **The Derived Gradient**: $Gradient = -2x(y - y_{predicted}) = -2x(Error)$.

### The Update Rule
**$$w_{new} = w_{old} - (Learning\_Rate \times Gradient)$$**

Let's see this calculation in code!

In [5]:
# Manual Gradient Descent Update Example
weight = 10.0      # Our initial guess: $10 per mile
learning_rate = 0.05

actual_fare = 25.0
miles = 2.0

print(f"Initial Guess ($10/mile): ${weight * miles}")

# 1. Calculate Error (Predicted - Actual)
error = (weight * miles) - actual_fare # 20 - 25 = -5
print(f"Error (Predicted - Actual): ${error:.2f}")

# 2. Calculate Gradient (miles * error)
gradient = miles * error
print(f"Gradient: {gradient:.2f}")

# 3. Update Weight (Move downhill)
weight = weight - (learning_rate * gradient)

print(f"\nNew  weight: ${weight:.2f}/mile")
print(f"New Prediction: ${weight * miles:.2f} (Much closer to $25!)")

Initial Guess ($10/mile): $20.0
Error (Predicted - Actual): $-5.00
Gradient: -10.00

New Learned Rate: $10.50/mile
New Prediction: $21.00 (Much closer to $25!)


## ⚙️ 4. Hyperparameters: The Control Knobs

Hyperparameters are settings we choose manually to control the learning process.

- 🎓 **Learning Rate (Step Size)**:
    - Too high: The model "overshoots" the valley and jumps around (Unstable).
    - Too low: The model takes too many tiny steps (Slow).
- 🔄 **Batch Size**: The number of examples the model looks at before updating the weights. Looking at *all* data is slow, so we use small "batches."
- 🔄 **Epochs**: One complete pass through the entire dataset.
- 🏁 **Convergence**: When the Loss stops decreasing significantly, the model has finished learning.

---
## 🛠️ 5. Real-World Implementation with TensorFlow

Now we will apply these concepts to thousands of Chicago Taxi trips.

In [ ]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

# Load Data
DATA_URL = 'https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv'
df = pd.read_csv(DATA_URL)
training_df = df.loc[:, ('TRIP_MILES', 'TRIP_SECONDS', 'FARE')].dropna()

print(f"Loaded {len(training_df):,} trips.")

# Explore Correlation: Does Miles really affect Fare?
print("Correlation with Fare:")
print(training_df.corr()['FARE'])

### Defining our Automated Solver

**Complex Functions Explained:**
- `tf.keras.layers.Dense(units=1)`: Creates our mathematical line ($y=wx+b$). A single-unit Dense layer is exactly Linear Regression.
- `optimizer=RMSprop`: This is our **Gradient Descent Engine**. It handles all the complex derivative math to update the weights automatically.

In [ ]:
def build_model(learning_rate):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=1, input_shape=(1,))
    ])
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=learning_rate),
        loss='mean_squared_error',
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )
    return model

def train_model(model, df, feature, label, epochs, batch_size):
    history = model.fit(x=df[feature], y=df[label], batch_size=batch_size, epochs=epochs, verbose=0)
    return history

## 🧪 6. Experiments: Observing Learning

Notice how the **RMSE (Root Mean Squared Error)** decreases over time. This is the model "walking down the hill" to minimize error.

In [ ]:
learning_rate = 0.01
epochs = 30
batch_size = 50

model = build_model(learning_rate)
history = train_model(model, training_df, 'TRIP_MILES', 'FARE', epochs, batch_size)

plt.figure(figsize=(10, 6))
plt.plot(history.history['root_mean_squared_error'])
plt.title('Loss Curve: Model Learning over time')
plt.xlabel('Epochs')
plt.ylabel('Error in Dollars ($)')
plt.grid(True)
plt.show()

## 🎯 7. Accuracy Check: Observed vs Predicted
Finally, let's see how close our predictions are to reality. Points on the red dashed line are perfect predictions.

In [ ]:
sample_df = training_df.sample(500)
predictions = model.predict(sample_df['TRIP_MILES'])

plt.figure(figsize=(10, 6))
plt.scatter(sample_df['FARE'], predictions, alpha=0.5, color='teal')
plt.plot([0, 80], [0, 80], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Fare ($)')
plt.ylabel('Predicted Fare ($)')
plt.title('Final Accuracy: Actual vs. Predicted')
plt.legend()
plt.show()